# Fraud Detection - Complete MLflow Pipeline

This notebook demonstrates the **full MLflow experiment tracking and model lifecycle** on **Red Hat OpenShift AI 3.3** for fraud detection.

## Pipeline Steps

1. **Configure MLflow** - Connect to the MLflow server (Developer Preview)
2. **Train with experiment tracking** - Log parameters, metrics, artifacts, and plots
3. **Train with autolog** - Compare multiple models with automatic logging
4. **Register in MLflow Model Registry** - Version and manage models
5. **Promote best model** - Transition to Production stage
6. **Register in RHOAI Model Registry** - Full metadata for deployment via RHOAI

## Prerequisites

- Workbench created in `fraud-detection-ml` with the custom image
- Feature Store `fraud_detection` selected in the workbench
- **Connection** `minio-data-connection` attached to the workbench (S3/MinIO credentials)
- **Connection** `mlflow-data-connection` attached to the workbench (MLflow tracking URI, S3 endpoint, TLS cert path)
- MLflow enabled in the DataScienceCluster (`mlflowoperator: Managed`)
- Model Registry `fraud-detection` available

In [ ]:
# Only needed if NOT using the custom workbench image
# !pip install -q mlflow boto3 scikit-learn pandas matplotlib skl2onnx onnxruntime model-registry feast[postgres]

## Step 1 - Configure MLflow

Connect to the MLflow server deployed on RHOAI 3.3 (Developer Preview).

**Required workbench configuration:**
- **Connection** `minio-data-connection` attached (provides `AWS_*` env vars)
- **Connection** `mlflow-data-connection` attached (provides `MLFLOW_TRACKING_URI`, `MLFLOW_S3_ENDPOINT_URL`, `MLFLOW_TRACKING_SERVER_CERT_PATH`)
- Service account token auto-mounted by Kubernetes (authentication)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import mlflow
from mlflow.tracking import MlflowClient

# --- Validate required env vars from Connections ---
required_vars = {
    "MLFLOW_TRACKING_URI": "Connection mlflow-data-connection",
    "MLFLOW_S3_ENDPOINT_URL": "Connection mlflow-data-connection",
    "MLFLOW_TRACKING_SERVER_CERT_PATH": "Connection mlflow-data-connection",
    "AWS_ACCESS_KEY_ID": "Connection minio-data-connection",
    "AWS_SECRET_ACCESS_KEY": "Connection minio-data-connection",
    "AWS_S3_ENDPOINT": "Connection minio-data-connection",
}
missing = [f"  {k} (source: {v})" for k, v in required_vars.items() if k not in os.environ]
if missing:
    raise RuntimeError(
        "Missing environment variables. Attach the Connections to your workbench:\n"
        + "\n".join(missing)
    )

# --- Kubernetes auth: service account token ---
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()
    print("Using service account token for MLflow authentication")
else:
    print("WARNING: No service account token found, MLflow auth may fail")

# TLS: MLFLOW_TRACKING_SERVER_CERT_PATH is set by the mlflow-data-connection
# pointing to /var/run/secrets/kubernetes.io/serviceaccount/ca.crt

# Set tracking URI and experiment
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

EXPERIMENT_NAME = "fraud-detection"
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print(f"MLflow Tracking URI : {os.environ['MLFLOW_TRACKING_URI']}")
print(f"MLflow version      : {mlflow.__version__}")
print(f"S3 endpoint         : {os.environ['AWS_S3_ENDPOINT']}")
print(f"TLS CA              : {os.environ['MLFLOW_TRACKING_SERVER_CERT_PATH']}")
print(f"Experiment          : {experiment.name} (ID: {experiment.experiment_id})")
print(f"Artifact location   : {experiment.artifact_location}")

## Data Preparation

Retrieve features from the Feast Feature Store and prepare the training dataset.

In [ ]:
import numpy as np
import pandas as pd
from feast import FeatureStore

np.random.seed(42)

# Connect to Feast
feast_config_dir = "/opt/app-root/src/feast-config"
config_files = [
    os.path.join(feast_config_dir, f)
    for f in os.listdir(feast_config_dir)
    if os.path.isfile(os.path.join(feast_config_dir, f))
]
store = FeatureStore(fs_yaml_file=config_files[0])
print(f"Feast project: {store.project}")

# Simulate transactions
now = pd.Timestamp.now()
customer_ids = [f"C{str(i).zfill(5)}" for i in range(1, 51)]
N = 500

entity_df = pd.DataFrame({
    "customer_id": np.random.choice(customer_ids, N),
    "event_timestamp": [now - pd.Timedelta(hours=np.random.randint(1, 24)) for _ in range(N)],
    "transaction_amount": np.round(np.random.exponential(200, N), 2),
    "is_foreign_transaction": np.random.choice([0, 1], N, p=[0.85, 0.15]),
})

feature_refs = [
    "customer_profile:age",
    "customer_profile:account_age_days",
    "customer_profile:credit_limit",
    "customer_profile:num_cards",
    "transaction_stats:avg_transaction_amount_30d",
    "transaction_stats:num_transactions_7d",
    "transaction_stats:num_transactions_1d",
    "transaction_stats:max_transaction_amount_7d",
    "transaction_stats:num_foreign_transactions_30d",
    "transaction_stats:num_declined_transactions_7d",
]


def compute_risk_features(df):
    df = df.copy()
    df["amount_ratio_to_avg"] = df["transaction_amount"] / df["avg_transaction_amount_30d"].clip(lower=1)
    df["amount_ratio_to_max"] = df["transaction_amount"] / df["max_transaction_amount_7d"].clip(lower=1)
    df["risk_score"] = (
        df["amount_ratio_to_avg"] * 0.4
        + df["amount_ratio_to_max"] * 0.3
        + df["is_foreign_transaction"] * 0.3
    )
    return df


# Retrieve historical features
print(f"Retrieving {N} historical features from Feast...")
training_df = store.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()
training_df = compute_risk_features(training_df)
training_df = training_df.dropna()

# Generate fraud labels
fraud_probability = 1 / (1 + np.exp(-(training_df["risk_score"] - 1.5) * 3))
training_df["is_fraud"] = (np.random.random(len(training_df)) < fraud_probability).astype(int)

# Model features
MODEL_FEATURES = [
    "age", "account_age_days", "credit_limit", "num_cards",
    "avg_transaction_amount_30d", "num_transactions_7d", "num_transactions_1d",
    "max_transaction_amount_7d", "num_foreign_transactions_30d",
    "num_declined_transactions_7d",
    "transaction_amount", "is_foreign_transaction",
    "amount_ratio_to_avg", "amount_ratio_to_max", "risk_score",
]

X = training_df[MODEL_FEATURES]
y = training_df["is_fraud"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset: {len(training_df)} samples, {len(MODEL_FEATURES)} features")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Fraud rate: {y.mean():.2%}")

## Step 2 - Train with MLflow Experiment Tracking

Train a RandomForest model with **manual logging** of all parameters, metrics, artifacts, and plots.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# --- Training with full manual MLflow logging ---
with mlflow.start_run(run_name="RandomForest-manual") as run:
    rf_run_id = run.info.run_id
    print(f"Run ID: {rf_run_id}")

    # 1. Log parameters
    rf_params = {"n_estimators": 100, "max_depth": 12, "min_samples_split": 5, "random_state": 42}
    mlflow.log_params(rf_params)
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_features", len(MODEL_FEATURES))
    mlflow.log_param("n_train_samples", len(X_train))
    mlflow.log_param("n_test_samples", len(X_test))
    mlflow.log_param("feature_store", "feast")
    mlflow.log_param("feast_project", store.project)

    # 2. Train
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

    # 3. Log metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall": recall_score(y_test, y_pred_rf),
        "f1_score": f1_score(y_test, y_pred_rf),
        "roc_auc": roc_auc_score(y_test, y_proba_rf),
    }
    mlflow.log_metrics(metrics)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()
    mlflow.log_metrics({"true_positives": tp, "true_negatives": tn,
                        "false_positives": fp, "false_negatives": fn})

    # 4. Log confusion matrix plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=ax,
                                            display_labels=["Legitimate", "Fraud"])
    ax.set_title("RandomForest - Confusion Matrix")
    plt.tight_layout()
    plt.savefig("/tmp/rf_confusion_matrix.png", dpi=100)
    mlflow.log_artifact("/tmp/rf_confusion_matrix.png", artifact_path="plots")
    plt.show()
    plt.close()

    # 5. Log ROC curve
    fig, ax = plt.subplots(figsize=(8, 6))
    RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=ax, name="RandomForest")
    ax.set_title("RandomForest - ROC Curve")
    plt.tight_layout()
    plt.savefig("/tmp/rf_roc_curve.png", dpi=100)
    mlflow.log_artifact("/tmp/rf_roc_curve.png", artifact_path="plots")
    plt.show()
    plt.close()

    # 6. Log feature importance plot
    importances = pd.Series(rf_model.feature_importances_, index=MODEL_FEATURES).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    importances.plot(kind="barh", ax=ax)
    ax.set_title("Feature Importance (RandomForest)")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("/tmp/rf_feature_importance.png", dpi=100)
    mlflow.log_artifact("/tmp/rf_feature_importance.png", artifact_path="plots")
    plt.show()
    plt.close()

    # 7. Log classification report as text artifact
    report = classification_report(y_test, y_pred_rf, target_names=["Legitimate", "Fraud"])
    with open("/tmp/rf_classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("/tmp/rf_classification_report.txt", artifact_path="reports")

    # 8. Log feature list as artifact
    with open("/tmp/features.txt", "w") as f:
        f.write("\n".join(MODEL_FEATURES))
    mlflow.log_artifact("/tmp/features.txt", artifact_path="metadata")

    # 9. Log tags
    mlflow.set_tags({
        "team": "fraud-detection",
        "stage": "experiment",
        "data_source": "feast-feature-store",
        "feast_views": "customer_profile,transaction_stats",
    })

    # 10. Log the sklearn model artifact
    mlflow.sklearn.log_model(rf_model, artifact_path="model")

    print(f"\n=== RandomForest Results ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print(f"  Confusion: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    print(f"\nAll logged to MLflow run: {rf_run_id}")

## Step 3 - Train with MLflow Autolog

Compare multiple models using **autolog**, which automatically captures parameters, metrics, and model artifacts.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Enable sklearn autolog
mlflow.sklearn.autolog(log_models=True, log_datasets=True)

results = {}

# --- GradientBoosting ---
with mlflow.start_run(run_name="GradientBoosting-autolog") as run:
    gb_run_id = run.info.run_id
    mlflow.set_tags({"team": "fraud-detection", "stage": "experiment", "data_source": "feast-feature-store"})

    gb_model = GradientBoostingClassifier(
        n_estimators=150, max_depth=8, learning_rate=0.1, random_state=42
    )
    gb_model.fit(X_train, y_train)
    y_pred_gb = gb_model.predict(X_test)
    y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

    results["GradientBoosting"] = {
        "run_id": gb_run_id,
        "model": gb_model,
        "f1": f1_score(y_test, y_pred_gb),
        "auc": roc_auc_score(y_test, y_proba_gb),
        "accuracy": accuracy_score(y_test, y_pred_gb),
    }
    print(f"GradientBoosting - F1: {results['GradientBoosting']['f1']:.4f}, AUC: {results['GradientBoosting']['auc']:.4f}")

# --- LogisticRegression ---
with mlflow.start_run(run_name="LogisticRegression-autolog") as run:
    lr_run_id = run.info.run_id
    mlflow.set_tags({"team": "fraud-detection", "stage": "experiment", "data_source": "feast-feature-store"})

    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)
    y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

    results["LogisticRegression"] = {
        "run_id": lr_run_id,
        "model": lr_model,
        "f1": f1_score(y_test, y_pred_lr),
        "auc": roc_auc_score(y_test, y_proba_lr),
        "accuracy": accuracy_score(y_test, y_pred_lr),
    }
    print(f"LogisticRegression - F1: {results['LogisticRegression']['f1']:.4f}, AUC: {results['LogisticRegression']['auc']:.4f}")

# Disable autolog for subsequent cells
mlflow.sklearn.autolog(disable=True)

# Add RandomForest results
results["RandomForest"] = {
    "run_id": rf_run_id,
    "model": rf_model,
    "f1": metrics["f1_score"],
    "auc": metrics["roc_auc"],
    "accuracy": metrics["accuracy"],
}

# --- Comparison ---
print("\n=== Model Comparison ===")
comparison = pd.DataFrame(results).T[["f1", "auc", "accuracy"]]
comparison = comparison.sort_values("f1", ascending=False)
print(comparison.to_string(float_format="{:.4f}".format))

BEST_MODEL_NAME = comparison.index[0]
BEST_RUN_ID = results[BEST_MODEL_NAME]["run_id"]
BEST_MODEL = results[BEST_MODEL_NAME]["model"]
print(f"\nBest model: {BEST_MODEL_NAME} (run: {BEST_RUN_ID})")

## Step 4 - Register in MLflow Model Registry

Register the best model in the **MLflow Model Registry** with a version and description.

In [ ]:
REGISTRY_MODEL_NAME = "fraud-detection"

# Register the best model from its run
model_uri = f"runs:/{BEST_RUN_ID}/model"
mv = mlflow.register_model(model_uri=model_uri, name=REGISTRY_MODEL_NAME)

print(f"Registered model: {mv.name}")
print(f"  Version : {mv.version}")
print(f"  Run ID  : {mv.run_id}")
print(f"  Source   : {mv.source}")

# Add version description
client.update_model_version(
    name=REGISTRY_MODEL_NAME,
    version=mv.version,
    description=(
        f"{BEST_MODEL_NAME} trained on Feast features for fraud detection. "
        f"F1={results[BEST_MODEL_NAME]['f1']:.4f}, "
        f"AUC={results[BEST_MODEL_NAME]['auc']:.4f}, "
        f"Accuracy={results[BEST_MODEL_NAME]['accuracy']:.4f}"
    ),
)

# Add model-level description
try:
    client.update_registered_model(
        name=REGISTRY_MODEL_NAME,
        description="Fraud detection model trained with Feast Feature Store on RHOAI 3.3",
    )
except Exception:
    pass  # already exists

# Set alias for easy loading
client.set_registered_model_alias(REGISTRY_MODEL_NAME, "champion", mv.version)

print(f"\nModel version {mv.version} registered with alias 'champion'")

## Step 5 - Promote Best Model

Transition the model version to **Production** stage and add tags for traceability.

In [ ]:
# Set model version tags
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "stage", "production")
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "approved_by", "data-science-team")
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "model_type", BEST_MODEL_NAME)
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "f1_score", f"{results[BEST_MODEL_NAME]['f1']:.4f}")
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "roc_auc", f"{results[BEST_MODEL_NAME]['auc']:.4f}")
client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "feast_features", ",".join(MODEL_FEATURES))

# Load model from registry to validate
loaded_model = mlflow.pyfunc.load_model(f"models:/{REGISTRY_MODEL_NAME}@champion")
test_preds = loaded_model.predict(X_test[:5])

print(f"=== Model Promoted ===")
print(f"  Model  : {REGISTRY_MODEL_NAME}")
print(f"  Version: {mv.version}")
print(f"  Alias  : champion")
print(f"  Tags   : stage=production, model_type={BEST_MODEL_NAME}")
print(f"\nValidation predictions: {test_preds}")
print(f"Actual labels:          {y_test.values[:5]}")

# List all versions
print(f"\n=== All Versions ===")
for v in client.search_model_versions(f"name='{REGISTRY_MODEL_NAME}'"):
    alias = ", ".join(v.aliases) if v.aliases else "-"
    print(f"  v{v.version}: run={v.run_id[:8]}... alias={alias}")

## Step 6 - Register in RHOAI Model Registry

Register the production model in the **RHOAI Model Registry** (Kubeflow-based) with full metadata.
This enables deployment via the RHOAI dashboard and KServe/OVMS.

Steps:
1. Export the model to ONNX format (for OVMS serving)
2. Upload to MinIO S3
3. Register in RHOAI Model Registry with all metadata

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

# Export best model to ONNX
initial_type = [("input", FloatTensorType([None, len(MODEL_FEATURES)]))]
onnx_model = convert_sklearn(BEST_MODEL, initial_types=initial_type)

onnx_path = "/tmp/fraud_model/1/model.onnx"
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"ONNX model exported: {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB)")

# Validate ONNX
session = ort.InferenceSession(onnx_path)
onnx_preds = session.run(None, {"input": X_test.iloc[:3].values.astype(np.float32)})
print(f"ONNX predictions : {onnx_preds[0].flatten()}")
print(f"Sklearn predictions: {BEST_MODEL.predict(X_test.iloc[:3])}")
print("ONNX validation OK")

In [ ]:
import boto3

# Upload ONNX model to MinIO (credentials from Data Connection env vars)
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

# Ensure bucket exists
try:
    s3.create_bucket(Bucket="models")
except Exception:
    pass

model_s3_key = "fraud-detection/1/model.onnx"
s3.upload_file(onnx_path, "models", model_s3_key)
print(f"Model uploaded to s3://models/{model_s3_key}")

In [ ]:
from model_registry import ModelRegistry

# Connect to the RHOAI Model Registry
registry = ModelRegistry(
    server_address="https://fraud-detection.rhoai-model-registries.svc.cluster.local",
    port=8443,
    author="fraud-detection-notebook",
    is_secure=True,
    user_token=open(SA_TOKEN_PATH).read().strip() if os.path.exists(SA_TOKEN_PATH) else None,
)

best_metrics = results[BEST_MODEL_NAME]

# Register with full metadata
rm = registry.register_model(
    "fraud-detection",
    uri="s3://models/fraud-detection/1/model.onnx",
    version="1.0.0",
    description=(
        f"Production fraud detection model ({BEST_MODEL_NAME}), "
        f"trained with Feast features, exported to ONNX. "
        f"F1={best_metrics['f1']:.4f}, AUC={best_metrics['auc']:.4f}"
    ),
    model_format_name="onnx",
    model_format_version="1",
    storage_key="minio-data-connection",
    storage_path="fraud-detection/1/model.onnx",
    metadata={
        # Metrics
        "accuracy": str(round(best_metrics["accuracy"], 4)),
        "f1_score": str(round(best_metrics["f1"], 4)),
        "roc_auc": str(round(best_metrics["auc"], 4)),
        # Model info
        "model_type": BEST_MODEL_NAME,
        "framework": "scikit-learn",
        "export_format": "onnx",
        "n_features": str(len(MODEL_FEATURES)),
        "features": ",".join(MODEL_FEATURES),
        # Data lineage
        "feast_project": store.project,
        "feast_feature_views": "customer_profile,transaction_stats",
        "training_samples": str(len(X_train)),
        "test_samples": str(len(X_test)),
        # MLflow traceability
        "mlflow_experiment": EXPERIMENT_NAME,
        "mlflow_run_id": BEST_RUN_ID,
        "mlflow_model_version": str(mv.version),
    },
)

print(f"=== Registered in RHOAI Model Registry ===")
print(f"  Model    : {rm.name}")
print(f"  ID       : {rm.id}")

# Verify
reg_model = registry.get_registered_model("fraud-detection")
reg_version = registry.get_model_version("fraud-detection", "1.0.0")
reg_artifact = registry.get_model_artifact("fraud-detection", "1.0.0")

print(f"\n=== Verification ===")
print(f"  Registered Model : {reg_model.name} (ID: {reg_model.id})")
print(f"  Version          : {reg_version.name} (ID: {reg_version.id})")
print(f"  Artifact URI     : {reg_artifact.uri}")
print(f"  Format           : {reg_artifact.model_format_name}")
print(f"\nModel is ready for deployment via RHOAI dashboard (KServe + OVMS).")

## Summary

```
                         OpenShift AI 3.3
    +-----------------------------------------------------------+
    |                                                           |
    |   +-------------+        +-------------------------+      |
    |   |  Workbench   |        |  Feature Store (Feast)  |      |
    |   |  (Notebook)  |------->|  Online + Offline Store |      |
    |   +------+------+        +-------------------------+      |
    |          |                                                 |
    |          v                                                 |
    |   +------+------+   Step 2-3                               |
    |   |   MLflow      |   Experiment Tracking                  |
    |   |   (Dev Preview)|  Autolog + Manual                     |
    |   +------+------+                                          |
    |          |                                                 |
    |     Step 4-5                                               |
    |          v                                                 |
    |   +------+------+        +----------------+               |
    |   | MLflow Model |  6    | RHOAI Model    |               |
    |   | Registry     |------>| Registry       |               |
    |   +------+------+        +-------+--------+               |
    |          |                       |                         |
    |          v                       v                         |
    |   +-------------+        +----------------+               |
    |   |  S3 (MinIO)  |        | Model Serving  |               |
    |   |  Artifacts   |        | (KServe/OVMS)  |               |
    |   +-------------+        +----------------+               |
    +-----------------------------------------------------------+
```